In [1]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [2]:

# ============================================================
# 1. Load CSV (unchanged)
# ============================================================

SEED = 42
EPOCHS = 200

np.random.seed(SEED)
torch.manual_seed(SEED)


In [3]:

df = pd.read_csv("cyber_dataset_C.csv")


In [4]:

df


,prompt,tool_ground_truth,tool_predicted,pred_svm,source,row_id
0,The vendor gave us a jump-box address and temp...,SSHConnect,SSHConnect,SSHConnect,pilot,C0000
1,Can you establish a terminal session to the Ub...,SSHConnect,SSHConnect,SSHConnect,pilot,C0001
2,We moved the inventory appliance to a new subn...,NmapScan,NmapScan,NmapScan,pilot,C0002
3,Does 172.16.9.25 still advertise that old SSH ...,NmapScan,NmapScan,NmapScan,pilot,C0003
4,The change request says the database node shou...,PortScan,PortScan,PortScan,pilot,C0004
...,...,...,...,...,...,...
1495,Read the local journal around the fan-control ...,ReadSyslog,ReadAuthLog,ReadSyslog,batch495,C1510
1496,The VPN interface renamed itself on this lapto...,ReadSyslog,GetSystemInfo,ReadSyslog,batch495,C1511
1497,After we rolled out a new eBPF-based observabi...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1512
1498,Get the operating-system journal entries from ...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1513


In [5]:

encoder = LabelEncoder()

y = encoder.fit_transform(df["tool_ground_truth"])


In [6]:

num_classes = len(encoder.classes_)

num_classes 


11

In [7]:

class_to_id = {
    name: i for i, name in enumerate(encoder.classes_)
}

class_to_id


{'BlockIP': 0,
 'CheckFailedLogins': 1,
 'CheckVulnerability': 2,
 'GetSystemInfo': 3,
 'ListProcesses': 4,
 'ListeningPorts': 5,
 'NmapScan': 6,
 'PortScan': 7,
 'ReadAuthLog': 8,
 'ReadSyslog': 9,
 'SSHConnect': 10}

In [8]:

y_svm = np.array([
    class_to_id.get(name, -1)
    for name in df["pred_svm"]
])

y_svm.shape


(1500,)

In [9]:

y_svm


array([10, 10,  6, ...,  9,  9,  9])

In [10]:

y_llama = np.array([
    class_to_id.get(name, -1)
    for name in df["tool_predicted"]
])


In [18]:

# Unknown is encoded as -1.

set ( [  name for name in df["tool_predicted"]   ]  )


{'BlockIP',
 'CheckFailedLogins',
 'CheckVulnerability',
 'GetSystemInfo',
 'ListProcesses',
 'ListeningPorts',
 'NmapScan',
 'PortScan',
 'ReadAuthLog',
 'ReadSyslog',
 'SSHConnect',
 'Unknown'}

In [19]:

# ============================================================
# 2. Train / validation / test split
# ============================================================

indices = np.arange(len(df))

indices


array([   0,    1,    2, ..., 1497, 1498, 1499])

In [20]:


trainval_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)


In [21]:

print(  len(trainval_idx)  )
print(   len(test_idx   )  ) 


1200
300


In [22]:

test_idx


array([1343, 1241,  705,  100, 1048,   32,  948,  523,   93, 1478,  900,
        141,  205,  986, 1078, 1452,  288,  673,  622,   59, 1426,  382,
        203,  674,  867,  852, 1213, 1188, 1444, 1189, 1018,  353,  784,
        157,  191,  964,  642,   50,  313,  429,  495,  865, 1336, 1382,
       1345, 1211,  788, 1236,  984, 1025,  778, 1254,  344,   35, 1067,
         57,  548,  419,  547,  464,  559,  893,  672,  525,  480,  661,
       1378,  736,  428, 1015,  180,  173, 1178, 1233, 1391, 1232,  372,
        506, 1131,  776,  680, 1001,  633, 1172, 1403,  445, 1014,  824,
       1306,  829,  189,  877, 1356,  450,  337, 1002,  921,  592, 1044,
       1246,  385, 1486,  168, 1330,  139,  564, 1079,  238,  926,  760,
       1405,  568, 1144,  262,  832,    9,  312,  743,  781,   16,  297,
        556,   18,   83, 1135, 1140, 1274,  242, 1396,  426,  801, 1203,
       1231,  212,  111,  339,    1,  508,   94, 1427, 1416,  299,   30,
        860, 1351, 1137,  596,  620,  918,  107,  2

In [23]:

# Split remaining 1200 into 900 training + 300 validation
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=0.25,
    random_state=SEED,
    stratify=y[trainval_idx]
)


In [24]:

print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Testing:", len(test_idx))



Training: 900
Validation: 300
Testing: 300


In [25]:

# ============================================================
# 3. Features: SVM + Llama predictions (23 features)
# ============================================================

# SVM has 11 classes
svm_onehot = np.zeros((len(df), num_classes))

svm_onehot.shape


(1500, 11)

In [26]:

y_svm


array([10, 10,  6, ...,  9,  9,  9])

In [33]:

valid_svm = y_svm >= 0

print(  np.sum( valid_svm )  )
print(     len( valid_svm )  )


1500
1500


In [29]:


svm_onehot[ 
    np.where(valid_svm)[0], y_svm[valid_svm] 
] = 1


In [34]:

for thing in svm_onehot:
    print(  thing  )


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 0

In [42]:

svm_onehot.shape


(1500, 11)

In [43]:

# Llama has 11 classes + Unknown
llama_onehot = np.zeros((len(df), num_classes + 1))

llama_onehot.shape


(1500, 12)

In [44]:

llama_idx = np.where(y_llama >= 0, y_llama, num_classes)


In [45]:


print(     len(  llama_idx )  )


1500


In [46]:

'''
import numpy as np

y_llama_rc = np.array([2, -1, 4, 0, -1])
num_classes_rc = 11

llama_idx_rc = np.where(y_llama_rc >= 0, y_llama_rc, num_classes_rc)

print("Original:", y_llama_rc)
print("Modified:", llama_idx_rc)

'''


'\nimport numpy as np\n\ny_llama_rc = np.array([2, -1, 4, 0, -1])\nnum_classes_rc = 11\n\nllama_idx_rc = np.where(y_llama_rc >= 0, y_llama_rc, num_classes_rc)\n\nprint("Original:", y_llama_rc)\nprint("Modified:", llama_idx_rc)\n\n'

In [48]:


llama_onehot[ 
      np.arange(len(df)), llama_idx
] = 1


In [49]:

for thing in llama_onehot:
    print( thing )


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1.

In [50]:


X = np.hstack((svm_onehot, llama_onehot)).astype(np.float32)

X.shape


(1500, 23)

In [51]:

X_train = torch.tensor( X[train_idx] )
X_val   = torch.tensor( X[val_idx]   )
X_test  = torch.tensor( X[test_idx]  )

print("Input features:", X.shape[1])


Input features: 23


In [52]:

# ============================================================
# 4. Selector targets
# ============================================================

# 0 = Select SVM
# 1 = Select Llama, only when Llama is correct and SVM wrong

selector_target = np.where(
    (y_llama == y) & (y_svm != y),
    1,
    0
)


In [53]:

selector_target.shape


(1500,)

In [54]:


target_train = torch.tensor(
    selector_target[train_idx],
    dtype=torch.long
)


In [55]:

target_train.shape


torch.Size([900])

In [56]:

target_train


tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0,
        0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [57]:

# ============================================================
# 5. Neural selector
# ============================================================

class SelectorNet(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.net(x)


In [59]:



model = SelectorNet(  X.shape[1]  )




In [60]:

X.shape[1]


23

In [61]:

# ============================================================
# 6. Weighted loss
# ============================================================

counts = np.bincount(
    selector_target[train_idx],
    minlength=2
)


print( counts )


[830  70]


In [63]:

weights = len(train_idx) / (
    2.0 * np.maximum(counts, 1)
)

print(  weights  )


[0.54216867 6.42857143]


In [64]:

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32)
)


In [65]:

optimizer = optim.Adam(model.parameters(), lr=0.001)


In [66]:

print("Training selector counts:", counts)
print("Class weights:", weights)


Training selector counts: [830  70]
Class weights: [0.54216867 6.42857143]


In [67]:



# ============================================================
# Train
# ============================================================

for epoch in range(EPOCHS):

    model.train()
    optimizer.zero_grad()

    logits = model(  X_train  )

    loss = criterion(logits, target_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f}"
        )





Epoch  20 | Loss = 0.6814
Epoch  40 | Loss = 0.6619
Epoch  60 | Loss = 0.6271
Epoch  80 | Loss = 0.5707
Epoch 100 | Loss = 0.4971
Epoch 120 | Loss = 0.4166
Epoch 140 | Loss = 0.3351
Epoch 160 | Loss = 0.2574
Epoch 180 | Loss = 0.1965
Epoch 200 | Loss = 0.1570


In [70]:

model.eval()

with torch.no_grad():
    selector_pred = torch.argmax(model(X_test), dim=1).numpy()

ensemble_pred = np.where(
    selector_pred == 1,
    y_llama[test_idx],
    y_svm[test_idx]
)


In [71]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = y[test_idx]

print("Accuracy:", accuracy_score(y_true, ensemble_pred))

print("Macro F1:", f1_score(
    y_true, ensemble_pred, average="macro"
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, ensemble_pred))

print("Classification Report:")
print(classification_report(
    y_true, ensemble_pred,
    zero_division=0
))


Accuracy: 0.9
Macro F1: 0.8969308167926792
Confusion Matrix:
[[28  0  0  0  0  0  0  0  0  0  0]
 [ 0 24  0  1  0  0  0  0  2  0  0]
 [ 0  0 27  0  0  0  0  0  0  0  0]
 [ 0  0  0 27  0  0  0  0  0  0  0]
 [ 0  0  0  4 23  0  0  0  0  0  0]
 [ 0  0  0  0  0 14  2 11  0  0  0]
 [ 0  0  1  0  0  0 26  1  0  0  0]
 [ 1  0  1  1  0  0  2 22  0  0  0]
 [ 0  0  0  0  0  0  0  0 27  0  0]
 [ 0  0  0  1  0  1  0  0  1 24  0]
 [ 0  0  0  0  0  0  0  0  0  0 28]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        28
           1       1.00      0.89      0.94        27
           2       0.93      1.00      0.96        27
           3       0.79      1.00      0.89        27
           4       1.00      0.85      0.92        27
           5       0.93      0.52      0.67        27
           6       0.87      0.93      0.90        28
           7       0.65      0.81      0.72        27
           8       0.90      1.0